# Dereddening

Computes A_Ks (Ks-band extinction) for all stars in the 4 catalogs using 3D dust maps.

**Dust maps used:**
- Edenhofer 2023: preferred when available (valid ~69–1250 pc, all-sky)
- Bayestar 2019: fallback (limited southern coverage, good for >~1 kpc)

**Extinction conversion:** Wang & Chen (2019) Table 3: A_Ks/A_V = 0.078 ± 0.004

**Output:** `cf_data/extinction.csv` with A_Ks and A_Ks_err per star

## Setup

Run this cell once to install dustmaps and download map data (~few GB, takes time).

In [6]:
from astroquery.vizier import Vizier
import pandas as pd
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from pathlib import Path
import requests
from io import StringIO
import time

from dustmaps.config import config
from dustmaps.bayestar import BayestarQuery
from dustmaps.edenhofer2023 import Edenhofer2023Query

# Must match the directory set in the setup cell
config['data_dir'] = '/opt/anaconda3/lib/python3.12/site-packages/dustmaps/data'

CF_DATA = Path('cf_data')

Vizier.ROW_LIMIT = -1

## Load catalogs

In [7]:
train = pd.read_csv(CF_DATA / 'training_stars.csv', dtype={'source_id': 'Int64'})
val   = pd.read_csv(CF_DATA / 'validation_stars.csv', dtype={'source_id': 'Int64'})
mm    = pd.read_csv(CF_DATA / 'mmcoeval.csv', dtype={'source_id': 'Int64'})
cfrc  = pd.read_csv(CF_DATA / 'cf_rotator_catalog.csv', dtype={'GaiaDR3_ID': 'Int64'})

# Combine into a single table with a unified source_id column
dfs = []
for df, id_col in [(train, 'source_id'), (val, 'source_id'), (mm, 'source_id')]:
    dfs.append(df[[id_col, 'ra', 'dec', 'parallax', 'parallax_error']].rename(columns={id_col: 'source_id'}))

cfrc_sub = cfrc[['GaiaDR3_ID', 'ra', 'dec', 'parallax', 'parallax_error']].rename(columns={'GaiaDR3_ID': 'source_id'})
dfs.append(cfrc_sub)

stars = pd.concat(dfs, ignore_index=True).drop_duplicates(subset='source_id')

print(f'Total unique stars: {len(stars)}')

Total unique stars: 11021


In [8]:
## Fetch Bailer-Jones distances from VizieR
VIZIER_TAP_URL = 'https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync'

def query_bailer_jones(source_ids, chunk_size=500):
    """Query Bailer-Jones EDR3 geometric distances (I/352/gedr3dis)."""
    results = []
    chunks = [source_ids[i:i+chunk_size] for i in range(0, len(source_ids), chunk_size)]
    for i, chunk in enumerate(chunks):
        if i % 5 == 0:
            print(f'  Chunk {i+1}/{len(chunks)}')
        id_list = ','.join(str(x) for x in chunk)
        query = f'SELECT Source, rgeo, b_rgeo, B_rgeo, rpgeo, b_rpgeo, B_rpgeo FROM "I/352/gedr3dis" WHERE Source IN ({id_list})'
        try:
            resp = requests.post(VIZIER_TAP_URL,
                data={'REQUEST': 'doQuery', 'LANG': 'ADQL', 'FORMAT': 'csv', 'QUERY': query},
                timeout=60)
            if resp.status_code == 200 and resp.text.strip():
                df = pd.read_csv(StringIO(resp.text))
                if len(df) > 0:
                    results.append(df)
        except Exception as e:
            print(f'  Error on chunk {i+1}: {e}')
        time.sleep(0.5)
    if results:
        return pd.concat(results, ignore_index=True)
    return pd.DataFrame(columns=['Source', 'rgeo', 'b_rgeo', 'B_rgeo', 'rpgeo', 'b_rpgeo', 'B_rpgeo'])

query_ids = stars['source_id'].dropna().astype(int).unique().tolist()
print(f'Querying Bailer-Jones distances for {len(query_ids)} stars...')
bj = query_bailer_jones(query_ids)
bj['Source'] = bj['Source'].astype('Int64')
print(f'Retrieved {len(bj)} / {len(query_ids)} from Bailer-Jones catalog')

# Cast to float (CSV may produce object dtype with mixed NaN representations)
for col in ['rgeo', 'b_rgeo', 'B_rgeo', 'rpgeo', 'b_rpgeo', 'B_rpgeo']:
    bj[col] = pd.to_numeric(bj[col], errors='coerce')

# Prefer rpgeo (photogeometric) when available, else rgeo (geometric)
bj['dist_pc'] = np.where(np.isfinite(bj['rpgeo']), bj['rpgeo'], bj['rgeo'])
bj['dist_pc_err'] = np.where(
    np.isfinite(bj['rpgeo']),
    (bj['B_rpgeo'] - bj['b_rpgeo']) / 2,
    (bj['B_rgeo']  - bj['b_rgeo'])  / 2
)

# Merge into stars
bj_lookup = bj.set_index('Source')[['dist_pc', 'dist_pc_err']]
stars['dist_pc']     = stars['source_id'].map(bj_lookup['dist_pc'])
stars['dist_pc_err'] = stars['source_id'].map(bj_lookup['dist_pc_err'])

# Fallback: 1/parallax for stars not in Bailer-Jones
fallback_mask = stars['dist_pc'].isna() & stars['parallax'].notna() & (stars['parallax'] > 0)
stars.loc[fallback_mask, 'dist_pc'] = 1000.0 / stars.loc[fallback_mask, 'parallax']
stars.loc[fallback_mask, 'dist_pc_err'] = (
    1000.0 * stars.loc[fallback_mask, 'parallax_error'] / stars.loc[fallback_mask, 'parallax']**2
)

print(f'\ndist_pc available:       {stars["dist_pc"].notna().sum()} / {len(stars)}')
print(f'  Bailer-Jones:          {len(bj)}')
print(f'  1/parallax fallback:   {fallback_mask.sum()}')
print(f'  No distance:           {stars["dist_pc"].isna().sum()}')
print(f'Dist range (pc): {stars["dist_pc"].min():.1f} – {stars["dist_pc"].max():.1f}')

Querying Bailer-Jones distances for 11021 stars...
  Chunk 1/23
  Chunk 6/23
  Chunk 11/23
  Chunk 16/23
  Chunk 21/23
Retrieved 0 / 11021 from Bailer-Jones catalog

dist_pc available:       10891 / 11021
  Bailer-Jones:          0
  1/parallax fallback:   10891
  No distance:           130
Dist range (pc): 1.3 – 1181968.4


## Load dust maps

In [9]:
# Load Edenhofer 2023 — using integrated mean map (mean_and_std_healpix.fits, already downloaded)
# load_samples=True requires the 19 GB samples file; integrated=True uses the pre-integrated mean
edenhofer = Edenhofer2023Query(load_samples=False, integrated=True)

# Load Bayestar 2019
bayestar = BayestarQuery(max_samples=10)

num_dist_samples = 10

print('Dust maps loaded.')

Integrating extinction map (this might take a couple of minutes)...
Optimizing map for querying (this might take a couple of seconds)...


Loading pixel_info ...
Loading samples ...
Loading best_fit ...
Replacing NaNs in reliable distance estimates ...
Sorting pixel_info ...
Extracting hp_idx_sorted and data_idx at each nside ...
  nside = 64
  nside = 128
  nside = 256
  nside = 512
  nside = 1024
t = 21.513 s
  pix_info:   0.299 s
   samples:  11.743 s
      best:   2.355 s
       nan:   0.031 s
      sort:   7.018 s
       idx:   0.066 s
Dust maps loaded.


## Query dust maps

In [10]:
samples = {}

queryable = stars[stars['dist_pc'].notna() & stars['ra'].notna() & stars['dec'].notna()].reset_index(drop=True)
skipped = stars[stars['dist_pc'].notna() & (stars['ra'].isna() | stars['dec'].isna())]
print(f'Querying {len(queryable)} stars...')
print(f'Skipping {len(skipped)} stars with missing RA/Dec (will get A_Ks = NaN)')

for i, r in queryable.iterrows():
    if i % 500 == 0:
        print(f'  Processing {i} / {len(queryable)}')

    gid = str(r.source_id)
    dist = r.dist_pc
    dist_err = r.dist_pc_err

    # If parallax_error is missing, query at single central distance only
    if pd.notna(dist_err) and dist_err > 0:
        dist_samples = np.random.normal(dist, dist_err, num_dist_samples)
        dist_samples = np.clip(dist_samples, 10.0, None)
    else:
        dist_samples = np.array([max(dist, 10.0)])

    b_all_samples = []
    e_all_samples = []

    for d in dist_samples:
        coord = SkyCoord(r.ra * u.deg, r.dec * u.deg, distance=d * u.pc, frame='icrs')

        # Bayestar — returns samples, check QA flags
        try:
            E_b = bayestar(coord, mode='samples', return_flags=True)
            qa_flags = E_b[1]
            if (qa_flags[0] & qa_flags[1]):
                for e in E_b[0]:
                    b_all_samples.append(e)
        except Exception:
            pass

        # Edenhofer integrated — returns scalar total extinction up to distance d
        try:
            E_e = edenhofer(coord, mode='mean')
            if np.isfinite(E_e):
                e_all_samples.append(float(E_e))
        except Exception:
            pass

    samples[gid] = {
        'E_samples_Bayestar':  np.array(b_all_samples),
        'E_samples_Edenhofer': np.array(e_all_samples),
    }

print('Done querying dust maps.')

Querying 10891 stars...
Skipping 0 stars with missing RA/Dec (will get A_Ks = NaN)
  Processing 0 / 10891
  Processing 500 / 10891
  Processing 1000 / 10891
  Processing 1500 / 10891
  Processing 2000 / 10891
  Processing 2500 / 10891
  Processing 3000 / 10891
  Processing 3500 / 10891
  Processing 4000 / 10891
  Processing 4500 / 10891
  Processing 5000 / 10891
  Processing 5500 / 10891
  Processing 6000 / 10891
  Processing 6500 / 10891
  Processing 7000 / 10891
  Processing 7500 / 10891
  Processing 8000 / 10891
  Processing 8500 / 10891
  Processing 9000 / 10891
  Processing 9500 / 10891
  Processing 10000 / 10891
  Processing 10500 / 10891
Done querying dust maps.


## Compute summary statistics

In [11]:
for gid in samples:
    samples[gid]['E_Bayestar_med'] = np.nanmedian(samples[gid]['E_samples_Bayestar']) if len(samples[gid]['E_samples_Bayestar']) > 0 else np.nan
    samples[gid]['E_Bayestar_std'] = np.nanstd(samples[gid]['E_samples_Bayestar'])    if len(samples[gid]['E_samples_Bayestar']) > 0 else np.nan
    samples[gid]['E_Edenhofer_med'] = np.nanmedian(samples[gid]['E_samples_Edenhofer']) if len(samples[gid]['E_samples_Edenhofer']) > 0 else np.nan
    samples[gid]['E_Edenhofer_std'] = np.nanstd(samples[gid]['E_samples_Edenhofer'])    if len(samples[gid]['E_samples_Edenhofer']) > 0 else np.nan

## Convert E → E(B-V) → A_V → A_Ks

Conversion factors (same as reference notebook):
- E(B-V) = 0.829 × E_Edenhofer
- E(B-V) = 0.88  × E_Bayestar
- A_V = R_V × E(B-V), R_V = 3.1
- A_Ks/A_V = 0.078 ± 0.004 (Wang & Chen 2019, Table 3)

In [12]:
Rv = 3.1

# Wang & Chen (2019) Table 3
A_Ks_div_AV     = 0.078
A_Ks_div_AV_err = 0.004

rows = []

for gid, s in samples.items():

    # Select dustmap: prefer Edenhofer if it returned valid samples, else Bayestar
    eden_valid = np.isfinite(s['E_Edenhofer_med']) and len(s['E_samples_Edenhofer']) > 0
    baye_valid = np.isfinite(s['E_Bayestar_med'])  and len(s['E_samples_Bayestar']) > 0

    if eden_valid:
        EBV     = s['E_Edenhofer_med'] * 0.829
        EBV_err = s['E_Edenhofer_std'] * 0.829
        dustmap_used = 'Edenhofer'
    elif baye_valid:
        EBV     = s['E_Bayestar_med'] * 0.88
        EBV_err = s['E_Bayestar_std'] * 0.88
        dustmap_used = 'Bayestar'
    else:
        EBV = EBV_err = 0.0
        dustmap_used = 'none'

    A_V     = Rv * EBV
    A_V_err = Rv * EBV_err

    A_Ks = A_V * A_Ks_div_AV
    if A_V > 0:
        A_Ks_err = A_Ks * np.sqrt((A_V_err / A_V)**2 + (A_Ks_div_AV_err / A_Ks_div_AV)**2)
    else:
        A_Ks_err = 0.0

    rows.append({
        'source_id':    int(gid),
        'A_V':          A_V,
        'A_V_err':      A_V_err,
        'A_Ks':         A_Ks,
        'A_Ks_err':     A_Ks_err,
        'dustmap_used': dustmap_used,
    })

ext_df = pd.DataFrame(rows)

# Stars with no parallax or no RA/Dec get NaN
missing_stars = stars[stars['dist_pc'].isna() | stars['ra'].isna() | stars['dec'].isna()].copy()
missing_stars['flag'] = np.where(missing_stars['dist_pc'].isna(), 'no_parallax', 'no_coords')
missing_ids = missing_stars['source_id'].dropna().astype(int).tolist()
missing_flags = missing_stars[missing_stars['source_id'].notna()].set_index(
    missing_stars[missing_stars['source_id'].notna()]['source_id'].astype(int)
)['flag']

missing_df = pd.DataFrame({
    'source_id':    missing_ids,
    'A_V':          np.nan,
    'A_V_err':      np.nan,
    'A_Ks':         np.nan,
    'A_Ks_err':     np.nan,
    'dustmap_used': [missing_flags.get(sid, 'no_parallax') for sid in missing_ids],
})

ext_df = pd.concat([ext_df, missing_df], ignore_index=True)

print(f'Total stars in extinction table: {len(ext_df)}')
print(f'dustmap_used counts:')
print(ext_df['dustmap_used'].value_counts())
print(f'\nA_Ks stats:')
print(ext_df['A_Ks'].describe())

Total stars in extinction table: 11021
dustmap_used counts:
dustmap_used
Edenhofer      9433
none            927
Bayestar        531
no_parallax     130
Name: count, dtype: int64

A_Ks stats:
count    10891.000000
mean         0.022606
std          0.026193
min          0.000000
25%          0.003719
50%          0.013230
75%          0.032860
max          0.234365
Name: A_Ks, dtype: float64


## Save

In [13]:
ext_df.to_csv(CF_DATA / 'extinction.csv', index=False)
print(f'Saved cf_data/extinction.csv ({len(ext_df)} rows)')

Saved cf_data/extinction.csv (11021 rows)
